# Basics

**🌐 언어:** [← English](/basics-en) | **한국어**

<small><em>작성자 김정현 · <a href="https://github.com/jhkimon">GitHub</a> · <a href="https://www.linkedin.com/in/%EC%A0%95%ED%98%84-%EA%B9%80-883a442b2/?skipRedirect=true">LinkedIn</a></em></small>

In [1]:
from IPython.display import HTML
HTML('''<iframe width="560" height="315" src="https://www.youtube.com/embed/eUBoSXqsrOE?si=KBHrZhPsrTX5L821" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>''')

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

## 약을 먹으면 입원 일수가 줄어드는가?

> **약을 먹으면 진짜 빨리 낫는 걸까요?**

이 질문은 우리의 삶과 아주 밀접한 문제입니다. 내가 아플 때 먹는 이 약이 입원 일수를 줄이는지, 병원에서 어떤 치료를 표준으로 삼을지와 같은 결정에 직접 영향을 미칩니다. 잘못된 답을 내리면 효과 있는 치료를 놓치거나, 효과 없는 치료를 믿고 사용하는 상황이 생길 수 있습니다.

이제 한 가지 상황을 떠올려 보겠습니다.

중증 환자는 원래 더 아픈 경우가 많아 입원 일수가 길고, 그만큼 약도 더 많이 복용하는 경향이 있습니다. 반대로 경증 환자는 상대적으로 덜 아파서 입원 일수가 짧고 약도 덜 복용합니다.

이런 상황에서 우리가 관찰하는 데이터는 과연 "약의 효과"를 보여주고 있을까요? 이런 상황에서 어떻게 하면 데이터를 활용해서 약의 효과를 왜곡되지 않게 계산할 수 있을까요?

### 인과추론 다시 살펴보기

우선 관심 있는 질문을 인과추론의 언어로 표현해 보겠습니다. 우리는 단순히 "먹은 사람 vs 안 먹은 사람"이 아니라, "같은 사람이 약을 먹었을 때 vs 안 먹었을 때"를 비교하고 싶습니다. 하지만 두 값을 같은 사람에게서 동시에 관찰할 수 없으므로, 전체 집단에서의 평균 효과를 목표로 합니다.

즉, "전체 집단에서 모든 사람이 약을 복용했을 때의 평균 입원 일수와, 아무도 복용하지 않았을 때의 평균 입원 일수가 얼마나 차이 나는가?"를 묻는 것입니다.

$$
ATE = \mathbb{E}[Y_1 - Y_0]
$$

여기서 $Y_1$은 약을 복용했을 때의 입원 일수, $Y_0$는 복용하지 않았을 때의 입원 일수입니다. 인과추론에서는 이를 **평균 인과효과(ATE, Average Treatment Effect)** 라고 부릅니다.

가장 단순한 방법은 약을 먹은 사람과 먹지 않은 사람의 평균 입원 일수를 그냥 비교하는 것입니다. 이 단순 평균 비교는 약이 무작위로 배정된 경우(RCT)에는 타당합니다. 치료 할당이 무작위이면 약 복용 여부가 중증도 같은 교란변수와 독립이 되어, 두 집단을 직접 비교해도 인과효과를 추정할 수 있기 때문입니다.

하지만 지금 상황은 다릅니다. 약을 먹은 사람은 대부분 중증 환자들이고, 안 먹은 사람은 대부분 경증 환자들입니다. 중증 환자들은 원래 더 아프기 때문에 입원 일수가 긴데, 이 때문에 **약 때문이 아니라 중증도 때문에 입원 일수가 달라 보이는 것**입니다.

```mermaid
graph LR
    Treatment[약 복용 T] --> Outcome[입원 일수 Y]
    Severity[중증도 X] --> Treatment
    Severity --> Outcome
```

이처럼 약 복용 여부와 입원 일수 모두에 영향을 주는 변수를 **교란변수(confounder)** 라고 합니다. 교란변수는 우리가 알고 싶은 ATE를 왜곡시키는 원인입니다.

이것이 실제로 문제가 될 수 있는지 아래 상황으로 확인해 보겠습니다.

In [ ]:
drug_example = pd.DataFrame(dict(
    severity= ["M","M","M","M","M","M", "W","W","W","W"],
    drug=[1,1,1,1,1,0,  1,0,1,0],
    days=[7,7,7,7,7,8,  2,3,2,3]
))
drug_example

| 그룹 | 약 복용 (T=1) | 약 미복용 (T=0) |
|---|---|---|
| 중증 | 5명 | 1명 |
| 경증 | 2명 | 2명 |

현실에서는 $Y_0$와 $Y_1$을 동시에 관측할 수 없지만, 설명을 위해 두 값을 모두 알고 있다고 가정하겠습니다.

- 중증: $Y_1 = 7, Y_0 = 8 \Rightarrow Y_1 - Y_0 = -1$
- 경증: $Y_1 = 2, Y_0 = 3 \Rightarrow Y_1 - Y_0 = -1$

## 단순 평균 비교의 문제

단순 평균 비교를 해보겠습니다. 약 복용 집단의 평균은 $(5 \times 7 + 2 \times 2)/7 = 39/7$, 약 비복용 집단은 $(1 \times 8 + 2 \times 3)/3 = 14/3$입니다.

In [ ]:
naive_ate = drug_example.query("drug==1")["days"].mean() - drug_example.query("drug==0")["days"].mean()
print(f"Naive ATE: {naive_ate:.4f}")

$$
\hat{ATE}_{naive} = 39/7 - 14/3 \approx +0.90
$$

이 값은 이상합니다. 경증 환자도 중증 환자도 약을 먹으면 입원 일수가 1일씩 줄어듭니다. 모든 집단에서 효과가 $-1$인데, 단순 평균 비교는 $+0.90$으로 부호가 정반대로 뒤집혀 마치 약이 해로운 것처럼 나옵니다.

이는 약의 효과가 아니라 **집단 구성의 차이(중증도 분포 차이)** 가 섞여 왜곡된 결과입니다.

이것을 우리는 **심슨의 역설(Simpson's paradox)** 이라고 부릅니다. 각 부분집단에서는 일관되게 나타나던 경향이, 집단을 합치면 정반대로 뒤집히는 현상이죠. 여기서도 약은 경증 집단과 중증 집단 모두에서 입원 일수를 줄이지만, 둘을 합쳐 비교하면 오히려 해로워 보입니다. 약을 먹은 사람 대부분이 중증 환자이고, 중증 환자는 약과 무관하게 원래 입원 일수가 길기 때문입니다.

### 실제 ATE

입원 일수가 1일 줄어드는 중증환자가 6명, 1일 줄어드는 경증환자가 4명이므로:

$$
ATE = \frac{-1 \times 6 + (-1) \times 4}{10} = -1.0
$$

단순 평균 비교는 약의 효과를 작게 보이게 할 뿐 아니라, **부호까지 뒤집어** 효과 있는 약을 해로운 것처럼 보이게 만듭니다.

## 해결책: Inverse Probability Weighting (IPW)

문제의 근본 원인은 **약을 먹은 집단과 먹지 않은 집단의 중증도 구성이 달랐기 때문**입니다. 두 집단이 같은 증상 심각성 구성을 가진 것처럼 보이도록 데이터를 재구성하면 어떨까요?

핵심 아이디어는 단순합니다. **각 개인에게 가중치를 부여하여, 마치 두 집단의 증상 심각성이 동일했던 것처럼 데이터를 다시 만들어주는 것**입니다. 특정 중증도이 어떤 집단에 과도하게 많으면 그 영향력을 줄이고, 덜 포함되어 있으면 늘려주는 방식입니다.

우리가 원하는 건, 각 증상 심각성 안에서 "약 먹은 사람 = 안 먹은 사람"이 되도록 만드는 것입니다. 중증 환자는 원래 약을 많이 먹기 때문에 약 안 먹은 중증환자가 너무 적습니다. 이 부족한 그룹을 "부풀려야" 합니다. 그래서 확률의 역수를 곱합니다.

이를 구현한 대표적인 방법이 **Inverse Probability Weighting (IPW)** 입니다. 각 개인에게 다음과 같은 가중치를 부여합니다.

$$
w =
\begin{cases}
\dfrac{1}{P(T=1 \mid X)} & \text{if } T=1 \\[6pt]
\dfrac{1}{P(T=0 \mid X)} & \text{if } T=0
\end{cases}
$$

이는 "해당 중증도에서 그 사람이 실제로 받은 처치를 받을 확률의 역수"로, **드물게 관찰된 경우일수록 더 큰 가중치를 부여하는 방식**입니다.

두 가지 상태를 합쳐 보통 다음과 같이 씁니다.

$$
w = \frac{T}{P(T=1 \mid X)} + \frac{1-T}{P(T=0 \mid X)}
$$

$T = 1$이면 첫 번째 항만, $T = 0$이면 두 번째 항만 남습니다.

### 왜 이 가중치가 균형을 만들어주는가?

중증도별 치료 확률을 계산해 보면:

$$
\text{중증: } P(T=1 \mid X=M)=\frac{5}{6}, \quad P(T=0 \mid X=M)=\frac{1}{6}
$$

$$
\text{경증: } P(T=1 \mid X=W)=\frac{1}{2}, \quad P(T=0 \mid X=W)=\frac{1}{2}
$$

가중치는 이 값의 역수입니다.

- 중증 + 약 복용: $6/5$, 중증 + 미복용: $6$
- 경증 + 약 복용: $2$, 경증 + 미복용: $2$

중증 환자 중 약을 복용하지 않은 사람은 원래 1명뿐이지만, 가중치가 6이므로 **마치 6명 있는 것처럼** 취급됩니다. 반대로 약을 복용한 중증 환자 5명은 각각 $6/5$의 가중치를 가져 총합이 6으로 맞춰집니다.

결과적으로 중증 환자] 집단에서는 약 복용/미복용 그룹의 총 weight가 모두 6으로 균형을 이루고, 경증 환자도 마찬가지입니다. 이제 단순 평균 비교만으로 원하는 ATE를 구할 수 있습니다.

$$
\hat{ATE}_{IPW} = \frac{7 \times 6 + 2 \times 4}{6+4} - \frac{8 \times 6 + 3 \times 4}{6+4} = -1.0
$$

In [ ]:
ps = drug_example.groupby("severity")["drug"].mean()
drug_example["ps"] = drug_example["severity"].map(ps)
drug_example["w"] = (
    drug_example["drug"] / drug_example["ps"]
    + (1 - drug_example["drug"]) / (1 - drug_example["ps"])
)

ate_ipw = (
    (drug_example["drug"] * drug_example["days"] * drug_example["w"]).sum()
    / (drug_example["drug"] * drug_example["w"]).sum()
    - ((1 - drug_example["drug"]) * drug_example["days"] * drug_example["w"]).sum()
    / ((1 - drug_example["drug"]) * drug_example["w"]).sum()
)
print(f"IPW ATE: {ate_ipw:.4f}")

이렇게 재구성된 데이터는 실제로 관측된 데이터가 아닙니다. 중증도과 치료가 마치 독립인 것처럼 보이도록 만든 **가상의 모집단(pseudo-population)** 으로 해석할 수 있습니다. 이 가상 모집단에서는 더 이상 특정 중증도이 특정 치료를 더 많이 받는 구조가 없기 때문에, 무작위로 약 복용이 배정된 것과 같은 상황이 됩니다.

## 참고: 성향점수 (Propensity Score)

가중치를 만들 때 사용한 치료 확률 $e(X) = P(T=1 \mid X)$를 **성향점수(propensity score)** 라고 부릅니다. 각 개인이 자신의 공변량 $X$를 바탕으로 특정 처치를 받을 확률을 의미합니다.

지금 예시에서는 $X$가 하나뿐이라 집단 내 치료 비율을 단순히 세어서 $P(T \mid X)$를 직접 계산할 수 있었습니다.

하지만 실제 분석에서는 이 확률을 정확히 알 수 없는 경우가 대부분입니다. 공변량이 여러 개이거나 연속형 변수가 포함된 경우, 단순 비율 계산만으로는 성향점수를 구할 수 없습니다. 그래서 실제로는 로지스틱 회귀 같은 모델로 성향점수를 **추정**합니다.

성향점수를 $e(X)$로 정의하고 데이터로부터 $\hat{e}(X)$를 추정하면, IPW 가중치는 다음과 같이 씁니다.

$$
\hat{w}_i =
\frac{T_i}{\hat{e}(X_i)} + \frac{1 - T_i}{1 - \hat{e}(X_i)}
$$

다만 이 방법도 성향점수를 얼마나 잘 추정하느냐에 크게 의존합니다. 나이와 중증도의 상호작용이나 비선형 관계를 단순한 모형으로 추정하면 $\hat{e}(X)$가 실제 확률을 제대로 반영하지 못할 수 있습니다. 그러면 가중치도 부정확해지고, 재가중된 데이터에서도 균형이 충분히 맞지 않아 교란이 완전히 제거되지 않을 수 있습니다.

## 참고 자료

이 노트북은 Hernán & Robins (2020), Matheus Facure의 *Causal Inference for the Brave and True*, 그리고 CausalInferenceLab의 한국어 번역본을 주요 참고 자료로 작성되었습니다.

- **Hernán, M. A., & Robins, J. M. (2020)**. *Causal Inference: What If*. Chapman & Hall/CRC.  
  [https://miguelhernan.org/whatifbook](https://miguelhernan.org/whatifbook)

- **Facure, M. (2022)**. *Causal Inference for the Brave and True*. Online book.  
  [https://matheusfacure.github.io/python-causality-handbook/](https://matheusfacure.github.io/python-causality-handbook/)

- **CausalInferenceLab**. *Causal Inference for the Brave and True* 한국어 번역본.  
  [https://causalinferencelab.github.io/Causal-Inference-with-Python/](https://causalinferencelab.github.io/Causal-Inference-with-Python/)